# EC vs RSS: TSP QAOA (Ring) + PUCCD Chemistry (X/Y Paulis)

Compare **Efficient Contraction** (exact) vs **Right Suffix Sampling** (1k shots) for parameter-shift gradient on:
1. **TSP QAOA** — ring topology, deep transpiled circuit
2. **PUCCD Chemistry** — H4 molecule (8 qubits), Hamiltonian with X/Y/Z Pauli terms

In [ ]:
# Install dependencies
!pip install -q qiskit qiskit-optimization qiskit-nature[pyscf] hashable_list ordered_set
!pip install -q git+https://github.com/keunjunpark/TREV@real_form_autograd

In [ ]:
import torch, math, time, gc, itertools
import numpy as np
import matplotlib.pyplot as plt

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.optimization.gradients.batch_parameter_shift import (
    expectation_value_batch_efficient_contraction,
    expectation_value_batch_right_suffix,
    BatchParameterShiftGradient,
)
from TREV.optimization.optimization import minimize as trev_minimize
from TREV.optimization.optimizer import Optimizer
from TREV.measure.right_suffix_sampling import argmax_bitstring_tr_right_suffix

from qiskit.circuit.library import QAOAAnsatz
from qiskit import transpile as qk_transpile
from qiskit.transpiler import CouplingMap
from qiskit_optimization.applications import Tsp
from qiskit_optimization.converters import QuadraticProgramToQubo
from TREV.transpile import from_qiskit

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SHIFT = math.pi / 2
BASIS_GATES = ['swap', 'rzz', 'rx', 'h']
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

## Shared Helpers

In [ ]:
from TREV.transpile import from_qiskit, build_parameter_mapping

def normalize_ising(qubitOp):
    max_coeff = max(abs(float(c.real)) for c in qubitOp.coeffs)
    if max_coeff > 0:
        qubitOp = qubitOp / max_coeff
    return qubitOp, max_coeff

def build_trev_hamiltonian(qubitOp):
    pauli_strings, coefficients = [], []
    for elm in qubitOp:
        pauli_strings.append(str(elm.paulis[0][::-1]))
        coefficients.append(float(elm.coeffs[0].real))
    return Hamiltonian(len(pauli_strings[0]), pauli_strings, coefficients)

def get_distance_matrix(tsp_inst):
    G = tsp_inst.graph
    n = len(G.nodes)
    dist = np.zeros((n, n))
    for i, j, data in G.edges(data=True):
        w = data.get('weight', 1.0)
        dist[i][j] = w; dist[j][i] = w
    return dist

def solve_tsp_brute(dist):
    n = dist.shape[0]
    best_cost, best_tour = float('inf'), None
    for perm in itertools.permutations(range(n)):
        cost = sum(dist[perm[i], perm[(i+1)%n]] for i in range(n))
        if cost < best_cost:
            best_cost, best_tour = cost, list(perm)
    return best_cost, best_tour

def decode_tsp_bitstring(bits, n_cities, qubit_perm=None):
    if isinstance(bits, str): bits = [int(b) for b in bits]
    bits = list(bits)
    if qubit_perm is not None:
        inv_perm = [0]*len(qubit_perm)
        for i, p in enumerate(qubit_perm): inv_perm[p] = i
        bits = [bits[inv_perm[i]] for i in range(len(bits))]
    n = n_cities; N = n*n
    if len(bits) < N: return False, None
    matrix = np.array(bits[:N]).reshape(n, n)
    if not (np.all(matrix.sum(axis=1)==1) and np.all(matrix.sum(axis=0)==1)):
        return False, None
    tour = [int(np.argmax(matrix[:, t])) for t in range(n)]
    return True, tour

def compute_tour_cost(tour, dist):
    n = len(tour)
    return sum(dist[tour[i]][tour[(i+1)%n]] for i in range(n))

def run_vqe(circuit, theta0, hamil, measure_method, shots, n_iters, lr, label):
    """Full-parameter VQE."""
    grad = BatchParameterShiftGradient(
        shift=SHIFT, batch_size=None, shots=shots,
        measure_method=measure_method, depth=1)
    opt = Optimizer(torch.optim.Adam, {'lr': lr})
    theta_init = theta0.clone().detach().requires_grad_(True)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    new_theta, exp_vals, best_results, iter_times = trev_minimize(
        circuit, theta_init, hamil, opt, grad, n_iters,
        best_value_method='argmax_tr_noinv_BE')
    wall_time = time.time() - t0
    if hasattr(grad, '_gpu_pool') and grad._gpu_pool is not None:
        grad._gpu_pool.shutdown(); grad._gpu_pool = None
    exp_floats = [float(v.real) if hasattr(v, 'real') else float(v) for v in exp_vals]
    med_iter = np.median(iter_times[2:]) if len(iter_times) > 2 else np.median(iter_times)
    print(f'{label}: {wall_time:.1f}s total, {med_iter:.2f}s/iter, '
          f'final={exp_floats[-1]:.4f}, best={min(exp_floats):.4f}')
    return {'exp_values': exp_floats, 'best_results': best_results,
            'iter_times': iter_times, 'wall_time': wall_time}

def run_vqe_subspace(circuit, param_base, jacobian, hamil, x0,
                     measure_method, shots, n_iters, lr, label):
    """Subspace VQE: optimize in K-dim ansatz parameter space via gradient projection."""
    device = circuit.device
    pb = param_base.to(device)
    J = jacobian.to(device)
    x = x0.clone().detach().to(device)

    grad_est = BatchParameterShiftGradient(
        shift=SHIFT, batch_size=None, shots=shots,
        measure_method=measure_method, depth=1)

    x.requires_grad_(True)
    adam = torch.optim.Adam([x], lr=lr)

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    exp_values, best_results, iter_times = [], [], []
    t0 = time.time()

    with torch.no_grad():
        for epoch in range(n_iters):
            it = time.time()
            adam.zero_grad()
            full_theta = pb + J @ x
            full_grad = grad_est.run(full_theta.detach(), circuit, hamil)
            x.grad = J.T @ full_grad  # project gradient to subspace
            adam.step()
            if device == 'cuda': torch.cuda.synchronize()
            iter_times.append(time.time() - it)

            full_theta_eval = pb + J @ x
            ev = circuit.get_expectation_value(full_theta_eval, hamil, measure_method)
            exp_values.append(float(ev.real) if hasattr(ev, 'real') else float(ev))

            tensor = circuit.build_tensor(full_theta_eval)
            best_results.append(argmax_bitstring_tr_right_suffix(tensor))

            if epoch % 10 == 0:
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

    wall_time = time.time() - t0
    if hasattr(grad_est, '_gpu_pool') and grad_est._gpu_pool is not None:
        grad_est._gpu_pool.shutdown(); grad_est._gpu_pool = None
    med_iter = np.median(iter_times[2:]) if len(iter_times) > 2 else np.median(iter_times)
    print(f'{label}: {wall_time:.1f}s total, {med_iter:.2f}s/iter, '
          f'final={exp_values[-1]:.4f}, best={min(exp_values):.4f}')
    return {'exp_values': exp_values, 'best_results': best_results,
            'iter_times': iter_times, 'wall_time': wall_time}

print('Helpers loaded.')

---
## Part 1: TSP QAOA (Ring Topology)

In [ ]:
NC = 3; REPS = 2; RANK = 8; N_ITERS = 100; LR = 5e-3; SEED = 0

qubitOp, offset, tsp_inst = Tsp.create_random_instance(NC, seed=SEED), None, None
tsp_inst_obj = Tsp.create_random_instance(NC, seed=SEED)
qp = tsp_inst_obj.to_quadratic_program()
qubo = QuadraticProgramToQubo().convert(qp)
qubitOp, offset = qubo.to_ising()
qubitOp_n, scale = normalize_ising(qubitOp)
N = qubitOp_n.num_qubits
hamil = build_trev_hamiltonian(qubitOp_n)
dist = get_distance_matrix(tsp_inst_obj)
opt_cost, opt_tour = solve_tsp_brute(dist)

op_tensor = hamil.get_pauli_op_tensor()
n_x = (op_tensor==1).any(dim=1).sum().item()
n_y = (op_tensor==2).any(dim=1).sum().item()
print(f'TSP: {NC} cities, {N} qubits, {len(hamil.coefficients)} terms (X:{n_x}, Y:{n_y})')
print(f'Optimal: {opt_tour}, cost={opt_cost:.0f}')

qaoa = QAOAAnsatz(qubitOp_n, reps=REPS)
optimized = qk_transpile(qaoa, optimization_level=3, basis_gates=BASIS_GATES)
cm = CouplingMap.from_ring(N)
routed = qk_transpile(optimized, coupling_map=cm, optimization_level=1,
                      basis_gates=BASIS_GATES, seed_transpiler=SEED)

K = 2*REPS
gen = torch.Generator().manual_seed(SEED)
qaoa_x0 = 0.01 * torch.randn(K, generator=gen)
sorted_params = sorted(routed.parameters, key=lambda p: p.name)
bind_dict = {p: float(qaoa_x0[i]) for i, p in enumerate(sorted_params)}
qc_bound = routed.assign_parameters(bind_dict)
tsp_circuit, tsp_theta0 = from_qiskit(qc_bound, fuse_zz_swap=True, rank=RANK, device=DEVICE)
P = tsp_theta0.shape[0]
print(f'TREV: {P} params, rank={RANK}, depth={routed.depth()}')

In [ ]:
# Run EC
print('=== Efficient Contraction (exact) ===')
tsp_ec = run_vqe(tsp_circuit, tsp_theta0, hamil,
                 MeasureMethod.EFFICIENT_CONTRACTION, 0, N_ITERS, LR, 'EC')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Run RSS 1k
print('\n=== Right Suffix Sampling (1k shots) ===')
tsp_rss = run_vqe(tsp_circuit, tsp_theta0, hamil,
                  MeasureMethod.RIGHT_SUFFIX_SAMPLING, 1000, N_ITERS, LR, 'RSS-1k')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Plot TSP results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
qperm = tsp_circuit.qubit_perm

def accuracy_curve(best_results, nc, qperm, dist, opt_cost):
    best_ratio = 0.0; ratios = []
    for bits in best_results:
        if isinstance(bits, str): bits = [int(b) for b in bits]
        elif hasattr(bits, 'tolist'): bits = bits.tolist()
        else: bits = list(bits)
        feasible, tour = decode_tsp_bitstring(bits, nc, qperm)
        if feasible and tour is not None:
            cost = compute_tour_cost(tour, dist)
            ratio = opt_cost / cost if cost > 0 else 0.0
            best_ratio = max(best_ratio, ratio)
        ratios.append(best_ratio)
    return ratios

ax = axes[0]
ax.plot(tsp_ec['exp_values'], 'b-', lw=2, label=f'EC [{tsp_ec["wall_time"]:.0f}s]')
ax.plot(tsp_rss['exp_values'], 'r--', lw=1.5, label=f'RSS-1k [{tsp_rss["wall_time"]:.0f}s]')
ax.set_xlabel('Iteration'); ax.set_ylabel('Expectation Value')
ax.set_title(f'TSP QAOA Energy (nc={NC}, rank={RANK})'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ec_acc = accuracy_curve(tsp_ec['best_results'], NC, qperm, dist, opt_cost)
rss_acc = accuracy_curve(tsp_rss['best_results'], NC, qperm, dist, opt_cost)
ax.plot(ec_acc, 'b-', lw=2, label='EC')
ax.plot(rss_acc, 'r--', lw=1.5, label='RSS-1k')
ax.axhline(1.0, color='green', ls=':', alpha=0.5, label='Optimal')
ax.set_xlabel('Iteration'); ax.set_ylabel('Performance Ratio')
ax.set_title('TSP Solution Accuracy'); ax.set_ylim(-0.05, 1.1); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'\n{"Method":<12s} {"Wall(s)":<10s} {"s/iter":<10s} {"Best E":<12s} {"Best Acc":<10s}')
print('-'*54)
for name, res, acc in [('EC', tsp_ec, ec_acc), ('RSS-1k', tsp_rss, rss_acc)]:
    med = np.median(res['iter_times'][2:]) if len(res['iter_times'])>2 else 0
    print(f'{name:<12s} {res["wall_time"]:<10.1f} {med:<10.2f} {min(res["exp_values"]):<12.4f} {max(acc):<10.3f}')

---
## Part 2: PUCCD Chemistry — H4 (8 qubits, X/Y Pauli terms)

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper, InterleavedQubitMapper
from qiskit_nature.second_q.circuit.library import PUCCD, HartreeFock
import scipy.sparse.linalg as spla

CHEM_BASIS = ['cx', 'swap', 'u']
mapper = InterleavedQubitMapper(JordanWignerMapper())

mol = {'atom': 'H 0 0 0; H 0 0 0.735; H 0 0 1.47; H 0 0 2.205', 'charge': 0, 'spin': 0}
driver = PySCFDriver(atom=mol['atom'], charge=mol['charge'], spin=mol['spin'], basis='sto3g')
problem = driver.run()
num_spatial = problem.num_spatial_orbitals
num_particles = problem.num_particles
N_chem = 2 * num_spatial
nuc_rep = problem.nuclear_repulsion_energy
second_q = problem.hamiltonian.second_q_op()
qubitOp_chem = mapper.map(second_q)

mat = qubitOp_chem.to_matrix(sparse=True)
eigvals, _ = spla.eigsh(mat, k=1, which='SA')
exact_energy = float(eigvals[0]) + nuc_rep

qubitOp_chem_n, chem_scale = normalize_ising(qubitOp_chem)
chem_hamil = build_trev_hamiltonian(qubitOp_chem_n)

op_t = chem_hamil.get_pauli_op_tensor()
print(f'H4: {N_chem} qubits, E_exact={exact_energy:.6f} Ha')
print(f'Hamiltonian: {len(chem_hamil.coefficients)} terms')
print(f'  X terms: {(op_t==1).any(dim=1).sum().item()}, '
      f'Y terms: {(op_t==2).any(dim=1).sum().item()}, '
      f'Z-only: {((op_t!=0).any(dim=1) & ~(op_t==1).any(dim=1) & ~(op_t==2).any(dim=1)).sum().item()}')

CHEM_RANK = 8; CHEM_ITERS = 200; CHEM_LR = 1e-2

# Build PUCCD ansatz and route for ring
hf = HartreeFock(num_spatial, num_particles, mapper)
ansatz = PUCCD(num_spatial, num_particles, mapper, initial_state=hf)
optimized = qk_transpile(ansatz, optimization_level=1, basis_gates=CHEM_BASIS)
cm = CouplingMap.from_ring(N_chem)
routed = qk_transpile(optimized, coupling_map=cm, optimization_level=1,
                      basis_gates=CHEM_BASIS, seed_transpiler=0)

K_chem = ansatz.num_parameters
print(f'PUCCD: {K_chem} ansatz params, {optimized.size()} gates, routed depth={routed.depth()}')

# Build subspace mapping: full_theta = param_base + jacobian @ x
chem_circuit, chem_pb, chem_J, chem_pnames = build_parameter_mapping(
    routed, fuse_zz_swap=False, rank=CHEM_RANK, device=DEVICE)
P = chem_J.shape[0]
print(f'Subspace mapping: {K_chem} PUCCD params -> {P} TREV params')

# Initial PUCCD params
gen = torch.Generator().manual_seed(0)
chem_x0 = 0.01 * torch.randn(K_chem, generator=gen)

In [ ]:
print('=== Efficient Contraction (exact, subspace) ===')
chem_ec = run_vqe_subspace(chem_circuit, chem_pb, chem_J, chem_hamil, chem_x0,
                           MeasureMethod.EFFICIENT_CONTRACTION, 0, CHEM_ITERS, CHEM_LR, 'EC')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n=== Right Suffix Sampling (1k shots, subspace) ===')
chem_rss = run_vqe_subspace(chem_circuit, chem_pb, chem_J, chem_hamil, chem_x0,
                            MeasureMethod.RIGHT_SUFFIX_SAMPLING, 1000, CHEM_ITERS, CHEM_LR, 'RSS-1k')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Energy convergence (in Hartree)
ax = axes[0]
ec_ha = [e * chem_scale + nuc_rep for e in chem_ec['exp_values']]
rss_ha = [e * chem_scale + nuc_rep for e in chem_rss['exp_values']]
ax.plot(ec_ha, 'b-', lw=2, label=f'EC [{chem_ec["wall_time"]:.0f}s]')
ax.plot(rss_ha, 'r--', lw=1.5, label=f'RSS-1k [{chem_rss["wall_time"]:.0f}s]')
ax.axhline(exact_energy, color='green', ls=':', alpha=0.5, label=f'Exact ({exact_energy:.4f} Ha)')
ax.set_xlabel('Iteration'); ax.set_ylabel('Energy (Ha)')
ax.set_title(f'H4 PUCCD Energy (rank={CHEM_RANK})'); ax.legend(); ax.grid(True, alpha=0.3)

# Per-iteration timing
ax = axes[1]
ec_t = chem_ec['iter_times'][2:] if len(chem_ec['iter_times'])>2 else chem_ec['iter_times']
rss_t = chem_rss['iter_times'][2:] if len(chem_rss['iter_times'])>2 else chem_rss['iter_times']
methods = ['EC (exact)', 'RSS (1k shots)']
medians = [np.median(ec_t), np.median(rss_t)]
colors = ['tab:blue', 'tab:red']
ax.barh(methods, medians, color=colors, edgecolor='black')
for i, v in enumerate(medians):
    ax.text(v + 0.005, i, f'{v:.3f}s', va='center')
ax.set_xlabel('Median Iteration Time (s)')
ax.set_title('Per-Iteration Cost')
plt.tight_layout(); plt.show()

print(f'\nExact energy: {exact_energy:.6f} Ha')
print(f'{"Method":<12s} {"Wall(s)":<10s} {"s/iter":<10s} {"Best E(Ha)":<14s} {"Error(mHa)":<12s}')
print('-'*58)
for name, res, ha in [('EC', chem_ec, ec_ha), ('RSS-1k', chem_rss, rss_ha)]:
    med = np.median(res['iter_times'][2:]) if len(res['iter_times'])>2 else 0
    best_ha = min(ha)
    err = abs(best_ha - exact_energy) * 1000
    print(f'{name:<12s} {res["wall_time"]:<10.1f} {med:<10.3f} {best_ha:<14.6f} {err:<12.2f}')